# Notebook 05: Rule-Based Dialogue Detector (Tier 2)
Evaluating Stage A (Count female characters $\ge 2$), Stage B (Female-Female conversations), and Stage C (Male-talk scoring thresholding tuned strictly on training films).

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
os.chdir(root)
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
import pandas as pd
from bechdel.data.cornell_parser import parse_cornell_corpus
from bechdel.detector.rules import DialogueDetector
from bechdel.detector.evaluator import evaluate_detector_stages

tier2_df = pd.read_parquet('data/processed/tier2_features.parquet')
c_data = parse_cornell_corpus(extract_dir='data/raw/cornell')
lines_map = dict(zip(c_data['lines']['line_id'], c_data['lines']['text']))

detector = DialogueDetector(male_talk_threshold=0.1)
results = []
for _, row in tier2_df.iterrows():
    c_id = row['cornell_id']
    f_chars = c_data['characters'][c_data['characters']['cornell_id'] == c_id]
    f_convs = c_data['conversations'][c_data['conversations']['cornell_id'] == c_id]
    res = detector.analyze_film(c_id, row['title_cornell'], f_chars, lines_map, f_convs)
    results.append(res)

det_eval = evaluate_detector_stages(results, tier2_df)
print('Overall Detector Metrics:', det_eval['overall'])
print('Stage A Metrics:', det_eval['stage_a'])
print('Stage B Metrics:', det_eval['stage_b'])
print('Stage C Metrics:', det_eval['stage_c'])

Overall Detector Metrics: {'overall_accuracy': 0.6881188118811881, 'overall_precision': 0.8144329896907216, 'overall_recall': 0.42245989304812837, 'overall_f1': 0.5563380281690141, 'overall_specificity': 0.9170506912442397, 'overall_cohen_kappa': 0.35119426954548927, 'overall_tn': 199, 'overall_fp': 18, 'overall_fn': 108, 'overall_tp': 79, 'overall_support': 404}
Stage A Metrics: {'stage_a_accuracy': 0.5420792079207921, 'stage_a_precision': 0.9945945945945946, 'stage_a_recall': 0.5, 'stage_a_f1': 0.6654611211573237, 'stage_a_specificity': 0.9722222222222222, 'stage_a_cohen_kappa': 0.1434007243386971, 'stage_a_tn': 35, 'stage_a_fp': 1, 'stage_a_fn': 184, 'stage_a_tp': 184, 'stage_a_support': 404}
Stage B Metrics: {'stage_b_accuracy': 0.5788043478260869, 'stage_b_precision': 0.9484536082474226, 'stage_b_recall': 0.38016528925619836, 'stage_b_f1': 0.5427728613569321, 'stage_b_specificity': 0.9603174603174603, 'stage_b_cohen_kappa': 0.26687573903655337, 'stage_b_tn': 121, 'stage_b_fp': 5, 

In [2]:
# Inspect sample of flagged conversations exported for hand checking
sample_df = pd.read_csv('reports/handcheck_sample.csv')
print(f'Loaded {len(sample_df)} verification sample rows.')
display(sample_df[['film_title', 'speaker_1', 'speaker_2', 'male_talk_score', 'text']].head(5))

Loaded 50 verification sample rows.


,film_title,speaker_1,speaker_2,male_talk_score,text
0,ghost world,ENID,REBECCA,0.1111,It's obviously him! I can't believe it!
1,jackie brown,JACKIE,MELANIE,0.1250,Mr. Walker's my buddy. Ask him about Ordell. T...
2,scream 2,DEBBIE SALT,GALE,0.1000,"Jesus, youre just as whacked as your son. Whe..."
3,bound,CORKY,VIOLET,0.1094,"Caesar's Mafia, isn't he? You have to ask? No...."
4,out of sight,ADELE,KAREN,0.1818,"He didn't hurt you or anything, did he? You me..."
